# 🦜🔗 LangChain Local AI — Practical Examples

> A hands-on notebook exploring LangChain with **local Ollama models** and **Google Gemini**.
> Covers chains, structured output, batch processing, image vision, agent memory, and human-in-the-loop.

---

## 📋 Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Basic Chain — Stream & Batch](#2-basic-chain--stream--batch)
3. [String Output Parser with Gemini](#3-string-output-parser-with-gemini)
4. [Structured Output with Pydantic](#4-structured-output-with-pydantic)
5. [Invoice Extraction (Single)](#5-invoice-extraction-single)
6. [Invoice Batch Processing](#6-invoice-batch-processing)
7. [Image Vision with Ollama](#7-image-vision-with-ollama)
8. [Multi-Step Pipeline with Gemini](#8-multi-step-pipeline-with-gemini)
9. [Agent Memory — Summarization Middleware](#9-agent-memory--summarization-middleware)
10. [Human-in-the-Loop Middleware](#10-human-in-the-loop-middleware)

---

## Requirements

- [Ollama](https://ollama.com/) installed and running locally
- Models pulled: `ollama pull llama3.2:1b` and `ollama pull moondream`
- A `.env` file containing:
  ```
  GOOGLE_API_KEY=your_key_here
  GEMINI_MODEL=gemini-2.0-flash
  OPENAI_API_KEY=your_key_here
  ```


---
## 1. Setup & Installation


In [1]:
# Install required packages (run once)
# !pip install langchain langchain-ollama langchain-google-genai langchain-openai \
#              langgraph pydantic python-dotenv

In [2]:
# --- Standard Library ---
import base64
import os

# --- Third-Party ---
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Literal, Optional

# --- LangChain Core ---
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.tools import tool

# --- LangChain Integrations ---
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

# --- LangGraph ---
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()
print("✅ All imports loaded successfully.")

✅ All imports loaded successfully.


---
## 2. Basic Chain — Stream & Batch

Build a simple **prompt → LLM → parser** chain using LCEL (LangChain Expression Language).  
Demonstrates `.stream()` for real-time output and `.batch()` for parallel queries.


In [8]:
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0.3,  # Lower = more factual, less creative
)

llm_Gemini = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0.3,
)

prompt = ChatPromptTemplate.from_template("""
Answer the following question as a helpful assistant.
Question: {input}
Answer:""")

chain = prompt | llm_Gemini | StrOutputParser()
print("✅ Chain ready.")

✅ Chain ready.


In [10]:
print("--- Streaming Response ---")
for chunk in chain.stream({"input": "Why do parrots talk?"}):
    print(chunk, end="", flush=True)

--- Streaming Response ---
Parrots talk primarily because they are **highly social, intelligent creatures** that rely on mimicry to survive and thrive in their natural environments. Here is a breakdown of why they do it:

### 1. Social Bonding (The "Flock" Mentality)
In the wild, parrots live in complex social groups. To maintain their place in the "flock," they need to communicate with other members. Because they are naturally inclined to mimic the sounds of their companions, they view their human owners as their "flock." When they imitate human speech, they are essentially trying to fit in and bond with you.

### 2. High Intelligence
Parrots are among the most intelligent birds in the world. They have a high capacity for observational learning, which means they learn by watching and listening to those around them. They don't just mimic sounds randomly; they often learn to associate specific sounds or words with specific outcomes (like saying "hello" when someone walks in the room or 

In [11]:
questions = [
    "What is the capital of France?",
    "What is the name of the largest and most intelligent mammal?",
]
responses = chain.batch(questions)
print("--- Batch Responses ---")
for q, r in zip(questions, responses):
    print(f"\nQ: {q}")
    print(f"A: {r}")

--- Batch Responses ---

Q: What is the capital of France?
A: The capital of France is Paris.

Q: What is the name of the largest and most intelligent mammal?
A: The answer depends on how you define "largest" and "most intelligent," as these titles are held by two different animals:

*   **The largest mammal** is the **blue whale**. It is the largest animal known to have ever existed on Earth, reaching lengths of up to 100 feet and weighing as much as 200 tons.
*   **The most intelligent mammal** (excluding humans) is widely considered to be the **chimpanzee** or the **bottlenose dolphin**. Chimpanzees share about 99% of our DNA and exhibit complex problem-solving and tool use, while dolphins demonstrate advanced communication skills, self-awareness, and social intelligence.

If you are looking for the largest animal that is also considered highly intelligent, the **sperm whale** is often cited; it has the largest brain of any creature to have ever lived and possesses complex social st

---
## 3. String Output Parser with Gemini

Shows how `StrOutputParser` cleans raw LLM output into a plain string.  
Without the parser, Gemini returns a full `AIMessage` object including metadata.  
The third cell shows the cleanest pattern: composing it directly into an LCEL chain.


In [12]:
# Model name loaded from .env (e.g. "gemini-3.1-flash-lite-preview")
gemini_model_name = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")
llm_gemini = ChatGoogleGenerativeAI(
    model=gemini_model_name,
    temperature=0.6,
)
template = PromptTemplate.from_template("Who is the President of country {country}")
print("✅ Gemini LLM ready.")

✅ Gemini LLM ready.


In [13]:
# Without parser: raw AIMessage — includes metadata, signatures, etc.
chat = template.invoke({"country": "United States"})
raw_result = llm_gemini.invoke(chat)
print("--- Raw output type ---")
print(type(raw_result))
print("--- Text content ---")
print(raw_result.content)

--- Raw output type ---
<class 'langchain_core.messages.ai.AIMessage'>
--- Text content ---
[{'type': 'text', 'text': 'The current President of the United States is **Joe Biden**.', 'extras': {'signature': 'EjQKMgG+Pvb7PiD/UP6wEOTetXhkfahTEbyJJcKaIFv8mT6hS6Q+KguTU7o3QskCQtAB7eyZ'}}]


In [14]:
# With parser: clean plain string
parser = StrOutputParser()
parsed_result = parser.invoke(raw_result)
print("--- Parsed output (with StrOutputParser) ---")
print(parsed_result)

--- Parsed output (with StrOutputParser) ---
The current President of the United States is **Joe Biden**.


In [15]:
# Best practice: compose into a single LCEL chain
gemini_chain = template | llm_gemini | parser
result = gemini_chain.invoke({"country": "India"})
print(result)

The current President of India is **Droupadi Murmu**. She assumed office on July 25, 2022.


---
## 4. Structured Output with Pydantic

Use `.with_structured_output()` to make the LLM return a validated **Pydantic object**  
instead of raw text — great for data extraction pipelines.


In [16]:
class ProductInfo(BaseModel):
    name: str      = Field(..., description="The name of the product")
    price: float   = Field(..., description="The price of the product in USD")
    category: str  = Field(..., description="The category of the product")
    in_stock: bool = Field(..., description="Whether the product is currently in stock")

product_llm = llm_gemini.with_structured_output(ProductInfo)
product = product_llm.invoke("What is the price and availability of the latest MacBook Pro 14?")

print(f"Product : {product.name}")
print(f"Price   : ${product.price}")
print(f"Category: {product.category}")
print(f"In Stock: {product.in_stock}")
print(f"\nJSON dump:\n{product.model_dump_json(indent=2)}")

Product : MacBook Pro 14-inch
Price   : $1999.0
Category: Electronics
In Stock: True

JSON dump:
{
  "name": "MacBook Pro 14-inch",
  "price": 1999.0,
  "category": "Electronics",
  "in_stock": true
}


---
## 5. Invoice Extraction (Single)

Extract structured data from raw invoice text using a **nested Pydantic schema**.  
Demonstrates `Literal` types, optional fields, and computed `@property` values.


In [17]:
class LineItem(BaseModel):
    description: str
    quantity: int     = Field(ge=1)
    unit_price: float = Field(ge=0)

    @property
    def total(self) -> float:
        return self.quantity * self.unit_price


class Invoice(BaseModel):
    invoice_number: str
    date: str
    vendor_name: str
    vendor_address: Optional[str] = None
    items: List[LineItem]
    subtotal: float
    tax: Optional[float] = None
    total: float
    payment_status: Literal["paid", "pending", "overdue"] = "pending"


# json_schema is most reliable for complex nested schemas with local models
invoice_llm = ChatOllama(model="llama3.2:1b", temperature=0)
structured_invoice_llm = llm_gemini.with_structured_output(Invoice, method="json_schema")
print("✅ Invoice schema and LLM ready.")

✅ Invoice schema and LLM ready.


In [18]:
invoice_text = """
Invoice #INV-2025-001
Date: January 15, 2025
From: Acme Corp, 123 Business St

Items:
- Widget Pro (5) @ $29.99 each
- Service Fee (1) @ $50.00

Subtotal: $199.95
Tax: $16.00
Total: $215.95

Status: Paid
"""
invoice = structured_invoice_llm.invoke(
    f"Extract all data from this invoice text accurately:\n{invoice_text}"
)
print(f"Invoice Number : {invoice.invoice_number}")
print(f"Vendor         : {invoice.vendor_name}")
print(f"Date           : {invoice.date}")
print(f"Vendor Address : {invoice.vendor_address or 'N/A'}")
print(f"Subtotal       : ${invoice.subtotal:.2f}")
print(f"Tax            : ${invoice.tax or 0:.2f}")
print(f"Total          : ${invoice.total:.2f}")
print(f"Status         : {invoice.payment_status}")
print("\nLine Items:")
for item in invoice.items:
    print(f"  - {item.description}: {item.quantity} x ${item.unit_price:.2f} = ${item.total:.2f}")

Invoice Number : INV-2025-001
Vendor         : Acme Corp
Date           : January 15, 2025
Vendor Address : 123 Business St
Subtotal       : $199.95
Tax            : $16.00
Total          : $215.95
Status         : paid

Line Items:
  - Widget Pro: 5 x $29.99 = $149.95
  - Service Fee: 1 x $50.00 = $50.00


---
## 6. Invoice Batch Processing

Scale extraction to process **multiple invoices** in a loop with error handling.


In [19]:
class InvoiceSummary(BaseModel):
    invoice_number: str
    date: str
    vendor_name: str
    items: List[LineItem]
    total: float
    payment_status: Literal["paid", "pending", "overdue"] = "pending"

# we using the same gemini LLM, but you can choose a different one if you want

batch_structured_llm = llm_gemini.with_structured_output(InvoiceSummary, method="json_schema")

invoices_data = [
    "Invoice #INV-001, Date: Jan 10, From: Apple Store. Item: iPhone 15 (1) @ $799. Total: $799. Status: Paid",
    "Invoice #INV-002, Date: Jan 12, From: Amazon. Item: USB-C Cable (2) @ $15. Total: $30. Status: Pending",
    "Invoice #6789, Date: Feb 01, From: Cloud Hosting. Item: Server Pro (1) @ $50. Total: $50. Status: Overdue",
    "Invoice #TX-99, Date: Feb 05, From: Local Cafe. Item: Coffee (10) @ $4.50. Total: $45. Status: Paid",
]

processed_invoices = []
print(f"--- Processing {len(invoices_data)} invoices ---\n")
for i, text in enumerate(invoices_data, 1):
    try:
        print(f"Processing Invoice {i}...")
        result = batch_structured_llm.invoke(f"Extract structured data from this invoice text: {text}")
        processed_invoices.append(result)
    except Exception as e:
        print(f"  ⚠️  Error on invoice {i}: {e}")

print("\n--- Results ---")
STATUS_ICON = {"paid": "✅", "pending": "⏳", "overdue": "🔴"}
for inv in processed_invoices:
    icon = STATUS_ICON.get(inv.payment_status, "")
    print(f"[{inv.invoice_number}] {inv.vendor_name} — ${inv.total} {icon}")

--- Processing 4 invoices ---

Processing Invoice 1...
Processing Invoice 2...
Processing Invoice 3...
Processing Invoice 4...

--- Results ---
[INV-001] Apple Store — $799.0 ✅
[INV-002] Amazon — $30.0 ⏳
[6789] Cloud Hosting — $50.0 🔴
[TX-99] Local Cafe — $45.0 ✅


---
## 7. Image Vision with Ollama

Send images to a local vision model (`moondream`) using Base64 encoding.  
Works fully **offline** — no API key needed.

> 💡 Replace `IMAGE_PATH` with a real image path on your machine.


In [20]:
def encode_image(image_path: str) -> str:
    """Convert a local image file to a Base64-encoded string."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def analyze_image(image_path: str, prompt: str = "What is in this image?") -> str:
    """Send a local image to the Moondream vision model and return its description."""
    vision_llm = ChatOllama(model="moondream")
    image_b64  = encode_image(image_path)
    ext        = image_path.rsplit(".", 1)[-1].lower()
    mime       = "image/gif" if ext == "gif" else f"image/{ext}"

    message = HumanMessage(content=[
        {"type": "text",      "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{image_b64}"}},
    ])
    return vision_llm.invoke([message]).content


print("✅ Vision helper functions defined.")

✅ Vision helper functions defined.


In [21]:
# ✏️  Update this path to any image on your machine
IMAGE_PATH = "path/to/your/image.jpg"

if os.path.exists(IMAGE_PATH):
    description = analyze_image(
        IMAGE_PATH,
        prompt="According to what you see, can you explain what is happening in this image step by step?",
    )
    print("--- Model Analysis ---")
    print(description)
else:
    print(f"⚠️  Image not found at '{IMAGE_PATH}'. Update IMAGE_PATH to run this cell.")

⚠️  Image not found at 'path/to/your/image.jpg'. Update IMAGE_PATH to run this cell.


---
## 8. Multi-Step Pipeline with Gemini

A two-step chain powered by **Google Gemini**:
1. Ask the model to **pick** a well-known movie
2. Ask it to **detail** that movie using the first response as input

Showcases deeply nested schemas and chaining structured calls.

> 🔑 Requires `GOOGLE_API_KEY` in your `.env` file.


In [22]:
class MovieRef(BaseModel):
    title: str          = Field(..., description="Official movie title")
    year: Optional[int] = Field(None,  description="Release year if known")

class MoviePick(BaseModel):
    movie: MovieRef
    why_known: str    = Field(..., description="Short reason why this movie is well-known")
    confidence: float = Field(..., ge=0, le=1, description="Confidence score 0–1")

class PersonRole(BaseModel):
    name: str
    role: str = Field(..., description="Character name or production role")

class MovieMeta(BaseModel):
    genres: List[str]
    language: Optional[str] = None
    country: Optional[str]  = None
    runtime_minutes: Optional[int] = None

class MovieRatings(BaseModel):
    imdb: Optional[float] = Field(None, ge=0, le=10)
    rotten_tomatoes: Optional[int] = Field(None, ge=0, le=100)

class MovieDetails(BaseModel):
    movie: MovieRef
    plot: str
    themes: List[str]
    key_people: List[PersonRole]
    metadata: MovieMeta
    ratings: MovieRatings

print("✅ Movie schemas defined.")

✅ Movie schemas defined.


In [23]:
gemini = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL", "gemini-2.0-flash"),
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0,
)

# Step 1: Pick a movie
pick_llm  = gemini.with_structured_output(MoviePick)
movie_pick = pick_llm.invoke(
    "Choose ONE real, famous movie you are very confident you know well. "
    "Return valid structured data only."
)

# Step 2: Get full details for that same movie
details_llm   = gemini.with_structured_output(MovieDetails)
movie_details  = details_llm.invoke(
    f"Provide accurate structured details for: {movie_pick.movie.title} ({movie_pick.movie.year})."
)

print(f"🎬 Picked: {movie_pick.movie.title} ({movie_pick.movie.year})")
print(f"   Confidence : {movie_pick.confidence:.0%}")
print(f"   Why known  : {movie_pick.why_known}")
print("\n--- MoviePick JSON ---")
print(movie_pick.model_dump_json(indent=2))
print("\n--- MovieDetails JSON ---")
print(movie_details.model_dump_json(indent=2))

🎬 Picked: The Godfather (1972)
   Confidence : 100%
   Why known  : Widely considered one of the greatest films in cinema history, directed by Francis Ford Coppola and starring Marlon Brando and Al Pacino.

--- MoviePick JSON ---
{
  "movie": {
    "title": "The Godfather",
    "year": 1972
  },
  "why_known": "Widely considered one of the greatest films in cinema history, directed by Francis Ford Coppola and starring Marlon Brando and Al Pacino.",
  "confidence": 1.0
}

--- MovieDetails JSON ---
{
  "movie": {
    "title": "The Godfather",
    "year": 1972
  },
  "plot": "The aging patriarch of an organized crime dynasty transfers control of his clandestine empire to his reluctant son.",
  "themes": [
    "Family",
    "Power",
    "Corruption",
    "Loyalty",
    "The American Dream"
  ],
  "key_people": [
    {
      "name": "Marlon Brando",
      "role": "Vito Corleone"
    },
    {
      "name": "Al Pacino",
      "role": "Michael Corleone"
    },
    {
      "name": "Francis Ford

---
## 9. Agent Memory — Summarization Middleware

As conversations grow, context windows fill up and costs increase.  
`SummarizationMiddleware` automatically compresses old messages into a summary  
once a trigger threshold is reached.

Two strategies:
- **Message-based** — summarize after N messages
- **Token-based** — summarize after N tokens (fraction of context window)


In [24]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver # 2026 Standard for InMemorySaver

load_dotenv()

# 1. Initialize the model using the Google AI Studio backend
# Ensure GOOGLE_API_KEY is in your .env file
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0
)

# 2. Configure the agent with Summarization Middleware
msg_agent = create_agent(
    model=llm,
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm, # Using the same model for summarization
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)

msg_config = {"configurable": {"thread_id": "summarize-messages"}}
print("✅ Message-based summarization agent ready using AI Studio.")

✅ Message-based summarization agent ready using AI Studio.


In [25]:
for q in questions:
    response = msg_agent.invoke({"messages": [HumanMessage(content=q)]}, msg_config)
    
    # Extract the content and ensure it's a string for formatting
    raw_answer = response["messages"][-1].content
    answer = str(raw_answer) if isinstance(raw_answer, list) else raw_answer
    
    # Remove newlines for cleaner table printing
    answer = answer.replace('\n', ' ').strip()
    
    msg_count = len(response["messages"])
    print(f"{q:<22} {answer[:23]:<25} {msg_count}")

What is the capital of France? [{'type': 'text', 'text   2
What is the name of the largest and most intelligent mammal? [{'type': 'text', 'text   4


In [26]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels in a city — returns a formatted listing."""
    return (
        f"Hotels in {city}:\n"
        "  1. Grand Hotel  — 5★  $350/night  (spa, pool, gym)\n"
        "  2. City Inn     — 4★  $180/night  (business center)\n"
        "  3. Budget Stay  — 3★  $75/night   (free wifi)"
    )
# 1. Manually initialize the LLM to force the use of AI Studio / API Key
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

# 2. Pass the 'llm' instance (NOT a string) to create_agent
token_agent = create_agent(
    model=llm,  # Using the instance we created above
    tools=[search_hotels],
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,  # Use the same instance for summarization
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ],
)
token_config = {"configurable": {"thread_id": "summarize-tokens"}}


def count_tokens(messages: list) -> int:
    """Approximate token count (4 chars ≈ 1 token)."""
    return sum(len(str(m.content)) for m in messages) // 4

print("✅ Token-based summarization agent ready.")

✅ Token-based summarization agent ready.


In [28]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

print(f"{'City':<12} {'~Tokens':>8}  {'Messages':>10}")
print("-" * 35)
for city in cities:
    response = token_agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=token_config,
    )
    tokens = count_tokens(response["messages"])
    msgs   = len(response["messages"])
    print(f"{city:<12} {tokens:>8}  {msgs:>10}")

City          ~Tokens    Messages
-----------------------------------
Paris             395           8
London            514           9
Tokyo             540           9
New York          516           9
Dubai             530           9
Singapore         519           6


---
## 10. Human-in-the-Loop Middleware

Pause agent execution before a tool runs, then let a human **approve**, **reject**, or **edit** the call.  
Critical for high-stakes operations — sending emails, database writes, financial transactions.

Three scenarios:
- **10a. Approve** — let the action proceed as-is
- **10b. Reject** — cancel the action
- **10c. Edit** — modify arguments before execution


In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

# --- Shared mock tools ---

@tool
def read_email(email_id: str) -> str:
    """Read an email by its ID."""
    return f"Email content for ID: {email_id}"


@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {recipient} with subject '{subject}'"


def build_hitl_agent(thread_id: str):
    agent = create_agent(
        model=llm,
        tools=[read_email, send_email],
        checkpointer=InMemorySaver(),
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},
                    "read_email": False,  # No human review for reads
                }
            )
        ],
    )
    config = {"configurable": {"thread_id": thread_id}}
    return agent, config

EMAIL_REQUEST = "Send email to john@test.com with subject 'Hello' and body 'How are you?'"
print("✅ HITL agent factory ready.")

✅ HITL agent factory ready.


### 10a. Approve — let the tool call execute as-is


In [30]:
agent, config = build_hitl_agent("hitl-approve")

# Step 1: Agent pauses before executing send_email
result = agent.invoke({"messages": [HumanMessage(content=EMAIL_REQUEST)]}, config=config)

if "__interrupt__" in result:
    print("⏸️  Agent paused — awaiting human approval...")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,
    )
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️  Agent paused — awaiting human approval...
✅ Result: [{'type': 'text', 'text': "OK. I've sent the email to john@test.com.", 'extras': {'signature': 'EjQKMgG+Pvb7XZ62JUz0qnX6fxQH8VaFtqnI9nAsWKXqAZBdxj00hbI5rv0tWYxZHDMUXzPp'}}]


### 10b. Reject — cancel the tool call


In [31]:
agent, config = build_hitl_agent("hitl-reject")

result = agent.invoke({"messages": [HumanMessage(content=EMAIL_REQUEST)]}, config=config)

if "__interrupt__" in result:
    print("⏸️  Agent paused — rejecting the tool call...")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject"}]}),
        config=config,
    )
    print(f"🚫 Result: {result['messages'][-1].content}")

⏸️  Agent paused — rejecting the tool call...
🚫 Result: [{'type': 'text', 'text': 'The request to send the email was rejected. Please let me know if you would like me to try again or if there is anything else I can help you with.', 'extras': {'signature': 'EjQKMgG+Pvb71JWijbBN8kpwain+0gpmBlPqFtsMXFw0ppF23qPr1lQyDQ2pnfQ2bsJeXWJw'}}]


### 10c. Edit — modify the arguments before the tool executes


In [32]:
agent, config = build_hitl_agent("hitl-edit")

result = agent.invoke(
    {"messages": [HumanMessage(content="Send an email to boss@company.com about the report")]},
    config=config,
)
if "__interrupt__" in result:
    print("⏸️  Agent paused — editing arguments before sending...")
    
    # Correct schema for 'edit' decision
    result = agent.invoke(
        Command(resume={
            "decisions": [{
                "type": "edit",
                "edited_action": {  # <--- This wrapper is required
                    "name": "send_email", # Specify the tool name
                    "args": {
                        "recipient": "boss@company.com",
                        "subject":   "Q1 Report — Final",
                        "body":      "Hi, please find the final Q1 report attached.",
                    }
                }
            }]
        }),
        config=config,
    )
    print(f"✏️  Result: {result['messages'][-1].content}")

⏸️  Agent paused — editing arguments before sending...
✏️  Result: [{'type': 'text', 'text': "OK. I've sent the email to your boss regarding the report.", 'extras': {'signature': 'EjQKMgG+Pvb75iYtPWgSlTV0ZdLT+H2TnGSUiHVSWGh7GeiipjmozAvrJOzX1k+YlPzUGsvm'}}]


# Document Loaders

## Text Loader

In [33]:
from langchain_community.document_loaders import TextLoader
path = "pyproject.toml"
loader = TextLoader(path)
print(f"✅ Loaded document from {path}:\n")
print(loader.load()[0].page_content)

✅ Loaded document from pyproject.toml:

[project]
name = "langchain-app"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.12"
dependencies = [
    "jq>=1.11.0",
    "langchain>=1.2.12",
    "langchain-chroma>=1.1.0",
    "langchain-community>=0.4.1",
    "langchain-google-genai>=4.2.1",
    "langchain-google-vertexai>=3.2.2",
    "langchain-ollama>=1.0.1",
    "langchain-text-splitters>=1.1.1",
    "pydantic>=2.12.5",
    "pydantic-settings>=2.13.1",
    "pypdf>=6.9.2",
    "python-dotenv>=1.2.2",
]



## PyPdf Loader

In [34]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "data/Machine Learning.pdf"  # Update with your PDF file path
try:
    pdf_loader = PyPDFLoader(pdf_path)
    print(f"✅ Loaded PDF document from {pdf_path}:\n")
    docs = pdf_loader.load()
    print(f"Document has {len(docs)} pages.")
    print(docs[0].page_content)
except Exception as e:
    print(f"❌ Error occurred while loading PDF document | {e}")

✅ Loaded PDF document from data/Machine Learning.pdf:

Document has 392 pages.
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


In [35]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

directory_path = "data\pdf_file"  # Update with your directory path

pdf_directory_loader = PyPDFDirectoryLoader(directory_path , glob="**/*.pdf")

docs = pdf_directory_loader.load()

print(f"Document has {len(docs)} pages.")
print(docs[0].page_content)

<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:3: SyntaxWarning: invalid escape sequence '\p'
C:\Users\hp\AppData\Local\Temp\ipykernel_3844\2007544838.py:3: SyntaxWarning: invalid escape sequence '\p'
  directory_path = "data\pdf_file"  # Update with your directory path


Document has 784 pages.
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


## CSV Loader

In [36]:
from langchain_community.document_loaders import CSVLoader

path = "data/ecomm.csv"

csv_loader = CSVLoader(path)

csv_docs= csv_loader.load()

print(len(csv_docs))
print(csv_docs[20].page_content)

418
InvoiceNo: 536367
StockCode: 48187
Description: DOORMAT NEW ENGLAND
Quantity: 4
InvoiceDate: 12-01-2010 08:34
UnitPrice: 7.95
CustomerID: 13047
Country: United Kingdom


## CSV Loader

In [37]:
from langchain_community.document_loaders import CSVLoader

path = "data/ecomm.csv"

csv_loader = CSVLoader(path)

csv_docs= csv_loader.load()

print(len(csv_docs))
print(csv_docs[20].page_content)

418
InvoiceNo: 536367
StockCode: 48187
Description: DOORMAT NEW ENGLAND
Quantity: 4
InvoiceDate: 12-01-2010 08:34
UnitPrice: 7.95
CustomerID: 13047
Country: United Kingdom


## JSON Loader

In [38]:
from langchain_community.document_loaders import JSONLoader

path = "data/sample.json"

json_laoder = JSONLoader(path, jq_schema=".", text_content=False)

json_docs = json_laoder.load()

print(len(json_docs))
print(json_docs[0].page_content)

1
{"name": "Alice", "age": 25, "skills": ["Python", "LangChain", "Data Analysis"], "is_active": true}


## Structured File Loader

In [39]:
from langchain_community.document_loaders import (
    JSONLoader, 
    PyPDFLoader, 
    TextLoader, 
    CSVLoader
)
from pathlib import Path

# Required dependencies: uv add jq pypdf
files = Path("data").glob("*")
all_docs = []

for file in files:
    try:
        if file.suffix == ".json":
            loader = JSONLoader(str(file), jq_schema=".", text_content=False)
        elif file.suffix == ".pdf":
            loader = PyPDFLoader(str(file))
        elif file.suffix == ".csv":
            loader = CSVLoader(str(file))
        elif file.suffix in [".txt", ".md"]:
            loader = TextLoader(str(file))
        else:
            print(f"Skipping unsupported file type: {file.name}")
            continue
            
        all_docs.extend(loader.load())
    except Exception as e:
        print(f"Failed to load {file.name}: {e}")

print(f"Total documents loaded: {len(all_docs)}")

Skipping unsupported file type: pdf_file
Total documents loaded: 819


In [40]:
print(all_docs[65].page_content)

InvoiceNo: 536373
StockCode: 82482
Description: WOODEN PICTURE FRAME WHITE FINISH
Quantity: 6
InvoiceDate: 12-01-2010 09:02
UnitPrice: 2.1
CustomerID: 17850
Country: United Kingdom


## Character Text Spliter

In [41]:
from langchain_text_splitters import CharacterTextSplitter

text = "Lancgchain makes it easy to build LLM-powerd app. It has text splitters to handle large documents."

ch_spliter = CharacterTextSplitter(
  separator="", chunk_size = 20, chunk_overlap = 3
)

chunks = ch_spliter.split_text(text)

print(len(chunks))
print(type(chunks))
print(chunks[0])
print(chunks[1])

6
<class 'list'>
Lancgchain makes it
it easy to build LLM


In [42]:
for i in chunks:
  print(i)

Lancgchain makes it
it easy to build LLM
LLM-powerd app. It h
t has text splitters
ers to handle large
ge documents.


## Recursive Character Text Spliter

In [43]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

path = "data/Machine Learning.pdf"
pdf_loader = PyPDFLoader(path)
pdf_docs = pdf_loader.load()

rec_spliter = RecursiveCharacterTextSplitter(
  separators=["\n\n", "\n", ".", " ",""], chunk_size = 500, chunk_overlap= 30
)

pdf_chunk = rec_spliter.split_documents(pdf_docs)

print(len(pdf_chunk))
print(pdf_chunk[10])


1663
page_content='NumPy                                                                                                                             7
SciPy                                                                                                                                 8
matplotlib                                                                                                                        9' metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2016-09-21T13:04:39+00:00', 'author': 'Andreas C. Müller and Sarah Guido', 'title': 'Introduction to Machine Learning with Python', 'trapped': '/False', 'moddate': '2020-08-19T07:09:16+02:00', 'source': 'data/Machine Learning.pdf', 'total_pages': 392, 'page': 4, 'page_label': 'iii'}


In [44]:
print(pdf_chunk[20].page_content)

Challenges in Unsupervised Learning                                                                        132
Preprocessing and Scaling                                                                                            132
Different Kinds of Preprocessing                                                                             133
Applying Data Transformations                                                                               134


## MarkdownHeader Text Spliter

In [45]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

path = "data/text.md"

md_laoder = TextLoader(path)
md_docs = md_laoder.load()

md_text = md_docs[0].page_content

md_spliter= MarkdownHeaderTextSplitter(
  headers_to_split_on=[("#","Title"), ("##", "Section"), ("###","Subsection")]
)

md_chunks = md_spliter.split_text(md_text)

print(len(md_chunks))
print(md_chunks[0].page_content)

9
Artificial Intelligence (AI) is transforming industries and redefining innovation.
In this guide, weâ€™ll explore its history, applications, and future.  
---


In [46]:
print(md_chunks[8].page_content)

The future of AI will include:
- **AGI (Artificial General Intelligence)** research
- Ethical and policy challenges
- Widespread automation of routine tasks


In [47]:
for i in md_chunks:
  print(i.page_content)

Artificial Intelligence (AI) is transforming industries and redefining innovation.
In this guide, weâ€™ll explore its history, applications, and future.  
---
The history of AI dates back to ancient myths of mechanical beings.
However, true AI research began in the **1950s**.
In 1956, the Dartmouth Conference marked the birth of AI as a field.
Key contributors included **John McCarthy** and **Marvin Minsky**.
AI faced multiple "winters" due to lack of funding and slow progress.  
---
AI is everywhere today.
Some of the most impactful applications include:
- Medical imaging analysis
- Predictive diagnostics
- Drug discovery
- Fraud detection
- Algorithmic trading
- Risk assessment
Self-driving cars, drones, and smart traffic management systems.  
---
The future of AI will include:
- **AGI (Artificial General Intelligence)** research
- Ethical and policy challenges
- Widespread automation of routine tasks


## Embedding 

In [48]:
from langchain_ollama.embeddings import OllamaEmbeddings
from dotenv import load_dotenv

load_dotenv()

embed = OllamaEmbeddings(model="mxbai-embed-large:latest")

embed.embed_query("I live in India")

[-0.05622794,
 -0.0033896705,
 -0.072400704,
 -0.01793094,
 -0.005372279,
 -0.020448362,
 0.051366154,
 0.024055647,
 0.036960486,
 0.04513499,
 0.006141674,
 0.031737346,
 0.017196331,
 -0.039005745,
 -0.042169783,
 -0.015822895,
 -0.036005605,
 -0.038094416,
 -0.031335752,
 0.03374004,
 0.009357405,
 0.047461092,
 -0.04196691,
 -0.053478934,
 0.023124507,
 0.02200328,
 0.0008659194,
 0.016456492,
 0.047731902,
 0.05775255,
 -0.030664565,
 0.018040841,
 0.0012861503,
 -0.022926858,
 -0.0073926602,
 0.031057343,
 0.015008048,
 0.002457201,
 -0.02802459,
 -0.029952684,
 -0.0029225617,
 -0.0009805423,
 0.018918604,
 -0.032101758,
 -0.02914986,
 -0.018760853,
 -0.05937424,
 -0.06550043,
 0.050850842,
 -0.00022460466,
 -0.004406077,
 0.022925554,
 -0.017618326,
 0.023679232,
 0.011375035,
 -0.027009124,
 -0.04272026,
 -0.010564284,
 -0.0027475413,
 0.02860133,
 -0.021002628,
 -0.001445346,
 0.020141624,
 -0.061733533,
 -7.999808e-05,
 0.0057362425,
 0.017022079,
 0.045586996,
 0.015644915,

In [49]:
vector = embed.embed_query("I want to have my dinner")
print(len(vector))



1024


In [50]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embed = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

text = [
  "I live in India",
  "I am a Data Scientist"
]

embed.embed_documents(text)

[[0.006005775,
  -0.00037283517,
  0.0055366335,
  -0.07951073,
  -0.012545535,
  0.017833866,
  0.026039347,
  0.01638503,
  0.0065808604,
  -0.009673562,
  -0.009016902,
  -0.0112344045,
  -0.0032370915,
  0.053243894,
  0.16205543,
  -0.005228321,
  -0.0068068965,
  -0.0021054712,
  0.011720414,
  -0.007643013,
  -0.0062179426,
  0.021685803,
  0.011470408,
  0.002794022,
  -0.019653233,
  -0.01074262,
  7.729086e-06,
  0.0003501423,
  0.0431107,
  -0.002126566,
  -0.0077165454,
  -0.013962217,
  0.0007476204,
  0.014745658,
  0.01812638,
  0.019943047,
  0.023383152,
  -0.0063697756,
  0.005784069,
  0.0056669363,
  -0.008996635,
  0.00968875,
  0.008847133,
  -0.007527219,
  0.010029602,
  -0.007561899,
  0.0021128918,
  -0.013976088,
  -0.02509941,
  0.01222467,
  0.019215832,
  -0.0041259043,
  -0.02548735,
  -0.26168045,
  -0.0010678123,
  0.01193521,
  -0.010968553,
  -0.011716459,
  -0.003302763,
  -0.017655758,
  -0.013401796,
  0.013425556,
  -0.01649719,
  -0.016304929,
  

In [51]:
vector = embed.embed_documents(text)
print(len(vector))
print(len(vector[1]))

2
3072


## Embed PDF File

### Using Gemini-embedding-001

### Using 

In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv

load_dotenv()

path = "data/Air pollution.pdf"
loader = PyPDFLoader(path)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=30)
chunks = splitter.split_documents(docs)
text_content = [chunk.page_content for chunk in chunks]



embed = OllamaEmbeddings(model="mxbai-embed-large:latest")
embed.embed_documents(text_content)

[[-0.0109507525,
  0.021128047,
  -0.017406566,
  0.0063687656,
  0.03816814,
  -0.015487911,
  0.008518382,
  -0.03550771,
  0.02200768,
  0.031086326,
  0.03265926,
  -0.019409586,
  0.02879195,
  -0.03183215,
  -0.0501186,
  -0.034453038,
  -0.03805383,
  0.0026675358,
  -0.052041426,
  0.0020396854,
  -0.030922707,
  0.042555105,
  -0.026976803,
  -0.015756309,
  0.0093129305,
  0.020226683,
  0.034541756,
  0.009243252,
  0.006145439,
  0.09044379,
  -0.023209533,
  0.0075508878,
  0.016841331,
  -0.00906207,
  -0.027358776,
  -0.0076629985,
  0.030619698,
  -0.06450998,
  -0.010786485,
  -0.00018501336,
  0.023145504,
  -0.029804986,
  0.03023994,
  -0.03107043,
  -0.008578315,
  0.009520027,
  -0.028904138,
  0.03203571,
  0.0146412,
  0.06352579,
  0.024507938,
  -0.005322344,
  0.026053656,
  -0.01975843,
  -0.0062327995,
  -0.02155244,
  -0.047423545,
  0.05754106,
  0.02662099,
  0.009258706,
  0.024638304,
  -0.0076197837,
  0.02713621,
  -0.024810666,
  0.038050335,
  0.05

In [53]:
vector = embed.embed_documents(text_content)
print("chunk size", len(vector))
print("Chunk Dimention", len(vector[15]))

chunk size 23
Chunk Dimention 1024


## Choma DB

### Adding Text

In [9]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
# New 2026 Standard Import
from langchain_ollama import OllamaEmbeddings 

load_dotenv()

embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")

text_docs = ["Hello World", "World Economy"]

ch_text_db = Chroma.from_texts(
    texts=text_docs, 
    embedding=embeddings, 
    collection_name="My_text", 
    persist_directory="./Chroma_db" 
)

# ch_text_db.persist() # <-- REMOVED: No longer required in current versions.

print(f"✅ Vector DB created at: {os.path.abspath('Chroma_db')}")

✅ Vector DB created at: c:\Users\hp\Desktop\Langchain\Chroma_db


In [10]:
print(ch_text_db.get()['documents'])

['Hello World', 'World Economy']


In [11]:
text_docs2 = ["Data Science", "Machine Learning"]
ch_text_db.add_texts(texts=text_docs2)

print(ch_text_db.get()['documents'])

['Hello World', 'World Economy', 'Data Science', 'Machine Learning']


In [12]:
print(ch_text_db.get()['documents'])

['Hello World', 'World Economy', 'Data Science', 'Machine Learning']


In [13]:
text_docs2 = ["Data Science", "Machine Learning"]
ch_text_db.add_texts(texts=text_docs2)

print(ch_text_db.get()['documents'])

['Hello World', 'World Economy', 'Data Science', 'Machine Learning', 'Data Science', 'Machine Learning']


In [14]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

# Load environment variables from .env file
load_dotenv()

# 1. Initialize Embeddings
# text-embedding-004 is the recommended updated model for Google GenAI
embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")  # Using OllamaEmbeddings as per previous context

# 2. Prepare Documents
docs = [
    Document(page_content="Hello World", metadata={"source": "doc1"}),
    Document(page_content="World Economy", metadata={"source": "doc2"})
]

# 3. Initialize Chroma and add documents
# Note: persist_directory should use './' prefix for reliable Windows pathing.
# Manual .persist() is no longer required as of Chroma 0.4.x.
ch_doc_db = Chroma.from_documents(
    documents=docs, 
    embedding=embeddings, 
    collection_name="My_docs", 
    persist_directory="./Chroma_db"
)

# 4. Retrieve and Print Documents
print("Stored Documents:", ch_doc_db.get()["documents"])
print("Stored Metadata:", ch_doc_db.get()["metadatas"])


Stored Documents: ['Hello World', 'World Economy', 'Data Science', 'Machine Learning', 'Hello World', 'World Economy']
Stored Metadata: [{'source': 'doc1'}, {'source': 'doc2'}, {'source': 'doc3'}, {'source': 'doc4'}, {'source': 'doc1'}, {'source': 'doc2'}]


In [15]:
docs2 = [
  Document(page_content="Data Science", metadata={"source": "doc3"}),
  Document(page_content="Machine Learning", metadata={"source": "doc4"})
]

ch_doc_db.add_documents(documents=docs2)
print(ch_doc_db.get()["documents"])

['Hello World', 'World Economy', 'Data Science', 'Machine Learning', 'Hello World', 'World Economy', 'Data Science', 'Machine Learning']


### Similarity Search

In [16]:
result = ch_doc_db.similarity_search("World", k=4)
for i in result:
  print(i.page_content, i.metadata)

World Economy {'source': 'doc2'}
World Economy {'source': 'doc2'}
Hello World {'source': 'doc1'}
Hello World {'source': 'doc1'}


## FAISS

### Adding Text

In [ ]:
from langchain_community.vectorstores import FAISS # New 2026 Standard for FAISS
load_dotenv()
embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")
text_docs = ["Hello World 2", "World Economy 2"]

# FAISS is an in-memory store; it does not persist to disk unless manually saved
faiss_text_db = FAISS.from_texts(texts=text_docs, embedding=embeddings)

In [ ]:
result1 = list(faiss_text_db.similarity_search("World", k=2))
for d in result1:
  print(d.page_content)

World Economy 2
Hello World 2


In [ ]:
text_docs2 = ["Data Science 2 ", "Machine Learning 2"]

faiss_text_db2 = FAISS.from_texts(texts=text_docs2, embedding=embeddings)

In [ ]:
result2 = list(faiss_text_db2.similarity_search("Data", k=2))
for d in result2:
  print(d.page_content)

Data Science 2 
Machine Learning 2


In [ ]:
faiss_text_db2.merge_from(faiss_text_db)

<div class="alert alert-block alert-success">
**docstore._dict.values()**
is used as the **.get()** in faiss
</div>

In [ ]:
result3 = list(faiss_text_db2.docstore._dict.values())
for d in result3:
  print(d.page_content)

Data Science 2 
Machine Learning 2
Hello World 2
World Economy 2


### Adding Document

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv()

embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")  # Using OllamaEmbeddings as per previous context
  
docs = [
  Document(page_content="Hello World 2", metadata={"source": "doc1"}),
  Document(page_content="World Economy 2", metadata={"source": "doc2"})
  
]

faiss_doc_db = FAISS.from_documents(documents=docs, embedding=embeddings)


In [ ]:
result4 = list(faiss_doc_db.docstore._dict.values())
for d in result4:
  print(d.page_content)

Hello World 2
World Economy 2


In [ ]:
docs2 = [
  Document(page_content="Data Science 2", metadata={"source": "doc3"}),
  Document(page_content="Machine Learning 2", metadata={"source": "doc4"})  
]

faiss_doc_db2 = FAISS.from_documents(documents=docs2, embedding=embeddings)

faiss_doc_db2.merge_from(faiss_doc_db)

result5 = list(faiss_doc_db2.docstore._dict.values())
for d in result5:
  print(d.page_content)

Data Science 2
Machine Learning 2
Hello World 2
World Economy 2


In [ ]:
result6 =faiss_doc_db2.similarity_search("Science", k=2)
for i in result6:
  print(i.page_content, i.metadata)

Data Science 2 {'source': 'doc3'}
Machine Learning 2 {'source': 'doc4'}


In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

# Create 5 Document objects
docs = [
    Document(page_content="Python is a programming language", metadata={"id": 1}),
    Document(page_content="Java is also a programming language", metadata={"id": 2}),
    Document(page_content="Cats sleep often during the day", metadata={"id": 3}),
    Document(page_content="Dogs bark at strangers", metadata={"id": 4}),
    Document(page_content="Birds fly in the sky", metadata={"id": 5}),
]

In [ ]:
# Initialize embedding model
embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")
emb = embeddings.embed_documents([doc.page_content for doc in docs])

# Create FAISS vector store from documents
faiss_db = FAISS.from_documents(documents=docs, embedding=embeddings)
# Perform similarity search
query = "coding "
results = faiss_db.similarity_search(query, k=5)
# Print results
for res in results:
    print(f"Content: {res.page_content}, Metadata: {res.metadata}")



Content: Python is a programming language, Metadata: {'id': 1}
Content: Java is also a programming language, Metadata: {'id': 2}
Content: Birds fly in the sky, Metadata: {'id': 5}
Content: Cats sleep often during the day, Metadata: {'id': 3}
Content: Dogs bark at strangers, Metadata: {'id': 4}


In [ ]:
query= "which animals are domestic pets?"
results = faiss_db.similarity_search(query, k=5)
for res in results:
    print(f"Content: {res.page_content}, Metadata: {res.metadata}") 
    

Content: Dogs bark at strangers, Metadata: {'id': 4}
Content: Cats sleep often during the day, Metadata: {'id': 3}
Content: Birds fly in the sky, Metadata: {'id': 5}
Content: Python is a programming language, Metadata: {'id': 1}
Content: Java is also a programming language, Metadata: {'id': 2}
